In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.subsampling import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../data/data_final_noend.csv")

diagnosis_variables = []

with open("../data/diagnosis_final_noend.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final_noend.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients with regional ln recurrence = 1
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]

data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)

In [6]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [7]:
print("Train events:", survival_dataset.iloc[train_index]["Regional LN recurrence"].sum())
print("Test events:", survival_dataset.iloc[test_index]["Regional LN recurrence"].sum())

Train events: 27.0
Test events: 6.0


In [8]:
subsampling = survival_dataset.iloc[train_index].copy()

x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)

In [85]:
# Define your hyperparameter ranges

alpha_range = np.logspace(-4, -1, 20)  # 20 alpha values
l1_ratio_range = [1.0, 0.8, 0.5, 0.01, 0.001]  # From Lasso to Ridge
controls_per_case_range = [1, 2, 4, 10, 15]

# Include deviance alongside other metrics
metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

# Run grid search
results, best_params, summary_df = subsampling_grid_search(
    dataset=subsampling,
    X=x_train,
    y=y_train,
    cv_folds=sgkf_1,
    alpha_range=alpha_range,
    l1_ratio_range=l1_ratio_range,
    controls_per_case_range=controls_per_case_range,
    metrics=metrics,
    feature_columns=feature_columns,
    scaler=StandardScaler(),
    verbose=True
)

SUBSAMPLING GRID SEARCH
Controls per case: [1, 2, 4, 10, 15]
L1 ratios: [1.0, 0.8, 0.5, 0.01, 0.001]
Alphas: 20 values from 1.00e-04 to 1.00e-01
Total combinations: 500

[1/500] controls=1, l1=1.000, alpha=1.00e-04 (elapsed: 0.0min, est. remaining: 0.0min)
⚠️  Info: 7 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1  5 32 33 38 40 47]
   Features after removal: 41 (removed 7)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


  ⚠️ Configuration failed: Input estimate contains NaN.
[2/500] controls=1, l1=1.000, alpha=1.44e-04 (elapsed: 0.0min, est. remaining: 6.6min)
⚠️  Info: 7 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1  5 32 33 38 40 47]
   Features after removal: 41 (removed 7)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


  ⚠️ Configuration failed: Input estimate contains NaN.
[3/500] controls=1, l1=1.000, alpha=2.07e-04 (elapsed: 0.0min, est. remaining: 7.8min)
⚠️  Info: 7 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1  5 32 33 38 40 47]
   Features after removal: 41 (removed 7)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


... (143 outputs left out)

c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


  ⚠️ Configuration failed: Input estimate contains NaN.
[410/500] controls=15, l1=1.000, alpha=2.64e-03 (elapsed: 18.2min, est. remaining: 4.0min)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 47]
   Features after removal: 45 (removed 3)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 38]
   Features after removal: 45 (removed 3)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 47]
   Features after removal: 45 (removed 3)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


  ⚠️ Configuration failed: Input estimate contains NaN.
[411/500] controls=15, l1=1.000, alpha=3.79e-03 (elapsed: 18.3min, est. remaining: 4.0min)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 47]
   Features after removal: 45 (removed 3)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 38]
   Features after removal: 45 (removed 3)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 47]
   Features after removal: 45 (removed 3)
⚠️  Info: 1 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [33]
   Fe

In [86]:
import pandas as pd
import numpy as np

# results = your_nested_dictionary
flattened_data = []

for sample_ratio, rank_dict in results.items():
    for rank_ratio, alpha_dict in rank_dict.items():
        for alpha, df_stats in alpha_dict.items():
            
            if df_stats is not None and not df_stats.empty:
                # 1. Identify which rows are actual Folds (e.g., Fold_1, Fold_2...)
                fold_indices = [i for i in df_stats.index if str(i).startswith('Fold')]
                
                # 2. Count how many of these folds have NaN values
                # We check if any column (c-index, etc.) is NaN for that fold
                nan_count = df_stats.loc[fold_indices].isna().any(axis=1).sum()
                
                # 3. Extract the Mean and Std rows
                if 'Mean' in df_stats.index and 'Std' in df_stats.index:
                    means = df_stats.loc['Mean']
                    stds = df_stats.loc['Std']
                    
                    row = {
                        'sampling_ratio': sample_ratio,
                        'rank_ratio': rank_ratio,
                        'alpha': alpha,
                        'nan_folds_count': nan_count,  # New tracking column
                        'c_index_mean': means['c-index'],
                        'c_index_ipcw_mean': means['c-index-ipcw'],
                        'brier_score_mean': means['int-brier-score'],
                        'c_index_std': stds['c-index'],
                        'c_index_ipcw_std': stds['c-index-ipcw'],
                        'brier_score_std': stds['int-brier-score']
                    }
                    flattened_data.append(row)

# Create the DataFrame
df_final = pd.DataFrame(flattened_data)

# Reorder columns to put the metadata and nan_count at the front
meta_cols = ['sampling_ratio', 'rank_ratio', 'alpha', 'nan_folds_count']
other_cols = [c for c in df_final.columns if c not in meta_cols]
df_final = df_final[meta_cols + other_cols]

# Sort: Primary sort by fewest NaNs, Secondary sort by best IPCW score
df_final = df_final.sort_values(by=['nan_folds_count', 'c_index_ipcw_mean'], 
                               ascending=[True, False])

print(df_final.head())

     sampling_ratio  rank_ratio     alpha  nan_folds_count  c_index_mean  \
159               2         0.5  0.100000                0      0.811616   
19                1         1.0  0.100000                0      0.832436   
119               2         1.0  0.100000                0      0.835750   
139               2         0.8  0.100000                0      0.821803   
158               2         0.5  0.069519                0      0.806321   

     c_index_ipcw_mean  brier_score_mean  c_index_std  c_index_ipcw_std  \
159           0.760839          0.043943     0.145862          0.128029   
19            0.759909          0.067940     0.055048          0.117217   
119           0.758998          0.040161     0.083587          0.093163   
139           0.756980          0.040653     0.115896          0.110827   
158           0.756326          0.048457     0.143569          0.136389   

     brier_score_std  
159         0.013195  
19          0.017128  
119         0.011791  


In [ ]:
df_final.to_csv("../results/subsampling/cox_subsampling_grid_search_results_noend.csv", index=False)

# Train and Test Final Model with Best Hyperparameters

In [16]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/cox_subsampling_grid_search_results.csv")
# Select row with highest c_index_ipcw_mean
best_params = best_results.iloc[0] #.sort_values('c_index_ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  sampling_ratio: {best_params['sampling_ratio']}")
print(f"  rank_ratio: {best_params['rank_ratio']}")
print(f"  alpha: {best_params['alpha']}")
print(f"  c_index_ipcw_mean (CV): {best_params['c_index_ipcw_mean']:.4f}")
print()

Best hyperparameters:
  sampling_ratio: 2.0
  rank_ratio: 1.0
  alpha: 0.1
  c_index_ipcw_mean (CV): 0.7590



In [17]:
# Prepare training fold dataframe (subsampling variable was created earlier)
train_fold = subsampling.copy()

# Apply case-control subsampling
x_train_sub, y_train_sub, sub_dataset = sample_case_control(
    train_fold=train_fold,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_controls_per_case=int(best_params['sampling_ratio']),
    feature_columns=feature_columns,
    random_state=12
)

print(f"Original training samples: {len(x_train)}")
print(f"Subsampled training samples: {len(x_train_sub)}")
print(f"Test samples: {len(x_test)}")

Original training samples: 2376
Subsampled training samples: 81
Test samples: 594


In [18]:
# Train final model with best hyperparameters
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

scaler = StandardScaler()
x_train_sub_scaled = scaler.fit_transform(x_train_sub)

final_model = CoxnetSurvivalAnalysis(
    l1_ratio=best_params['rank_ratio'],
    alphas=[best_params['alpha']],  # Use the specific alpha value found during grid search
    fit_baseline_model=True
)

final_model.fit(x_train_sub_scaled, y_train_sub)
print("Model trained successfully!")

Model trained successfully!


In [19]:
# Test the model on test set
x_test_scaled = scaler.transform(x_test)

# Calculate c-index
c_index = final_model.score(x_test_scaled, y_test)

# Calculate c-index IPCW
tau = y_train_sub["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train_sub, y_test, final_model.predict(x_test_scaled), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test_scaled)

# Use robust time filtering
max_time = min(y_train_sub['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train_sub['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train_sub, y_test, preds_at_times, times)

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")

\nTest Set Performance:
  C-index: 0.7607
  C-index IPCW: 0.7391
  Integrated Brier Score: 0.0240


# Feature Importance for SHAP Analysis

In [20]:
'''# Extract feature importance from Cox model
# For Cox models, feature importance is based on the absolute value of coefficients
import pandas as pd
import numpy as np

# Get coefficients from the final Cox model
# Since we specified alphas=[value], we get a single set of coefficients
coefficients = final_model.coef_

# Handle different shapes (squeeze to remove alpha dimension if present)
if coefficients.ndim > 1:
    coefficients = coefficients.squeeze()

# IMPORTANT: Use the actual feature names from the training data
# The model was trained on x_train_sub_scaled, which came from x_train_sub
actual_features = x_train_sub.columns.tolist()

# Verify lengths match
if len(coefficients) != len(actual_features):
    print(f"Error: Coefficient length ({len(coefficients)}) doesn't match feature length ({len(actual_features)})")
    print(f"This should not happen with single alpha")
    # Fallback: use minimum length
    min_len = min(len(coefficients), len(actual_features))
    coefficients = coefficients[:min_len]
    actual_features = actual_features[:min_len]

# Create dataframe with proper feature mapping
coef_df = pd.DataFrame({
    'Feature': actual_features,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
})

# Sort by absolute value of coefficient (most important first)
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)

# Display top 20 most important features
print("Top 20 Most Important Features for Cox Model:")
print("="*60)
print(coef_df.head(20).to_string(index=False))
print(f"\nTotal features: {len(coef_df)}")
print(f"Total non-zero coefficients: {(coef_df['Coefficient'] != 0).sum()}")

# Store for SHAP analysis
top_features_cox = coef_df.head(20)['Feature'].tolist()
print(f"\nTop features stored in 'top_features_cox' variable")
print(f"All coefficients stored in 'coef_df' variable")'''

'# Extract feature importance from Cox model\n# For Cox models, feature importance is based on the absolute value of coefficients\nimport pandas as pd\nimport numpy as np\n\n# Get coefficients from the final Cox model\n# Since we specified alphas=[value], we get a single set of coefficients\ncoefficients = final_model.coef_\n\n# Handle different shapes (squeeze to remove alpha dimension if present)\nif coefficients.ndim > 1:\n    coefficients = coefficients.squeeze()\n\n# IMPORTANT: Use the actual feature names from the training data\n# The model was trained on x_train_sub_scaled, which came from x_train_sub\nactual_features = x_train_sub.columns.tolist()\n\n# Verify lengths match\nif len(coefficients) != len(actual_features):\n    print(f"Error: Coefficient length ({len(coefficients)}) doesn\'t match feature length ({len(actual_features)})")\n    print(f"This should not happen with single alpha")\n    # Fallback: use minimum length\n    min_len = min(len(coefficients), len(actual_feat

In [21]:
'''# Export feature importance to CSV
coef_df.to_csv("../results/features/cox_feature_importance.csv", index=False)
print("Feature importance exported to: ../results/features/cox_feature_importance.csv")'''

'# Export feature importance to CSV\ncoef_df.to_csv("../results/features/cox_feature_importance.csv", index=False)\nprint("Feature importance exported to: ../results/features/cox_feature_importance.csv")'